# Defended H-ResFL: AAMAS 2027 Experimental Suite — Colab & Kaggle (Free T4 GPU)

**Paper Title:** *Defended H-ResFL: Efficient, Fair, and Byzantine-Resilient Multi-Agent Coordination via Multi-Scale Residual Personalization*  
**Authors:** Nghiem Duc Khanh Nam\*, Hung Anh Nguyen\*, Leandro Soriano Marcolino (Lancaster Univ & VinUni)  
**Target:** AAMAS 2027 (Abstract: Oct 1, 2026 | Full Paper: Oct 8, 2026)  

### Platform Compatibility (Zero Setup)
- **Google Colab:** Select **Runtime → Change runtime type → T4 GPU**.
- **Kaggle:** In the right sidebar, set **Settings → Accelerator: GPU T4 x2 (or x1)** and toggle **Internet: On**.

### 4 Modular Jobs Architecture (~1.5 Hours Total on T4 GPU)
1. **Job 1 (35 min):** CIFAR-100 High-Cardinality (=100$) across 5 Regimes (IID, Mild $\alpha=1.0$, Moderate $\alpha=0.5$, Severe $\alpha=0.1$, Extreme $\alpha=0.05$).
2. **Job 2 (25 min):** CIFAR-100 Byzantine Multi-Attack Robustness (Label & Sign flipping at  \in [0, 0.3]$) with Skew-Calibrated Subspace Defense.
3. **Job 3 (20 min):** 50-Client Scalability Benchmark (=0.20$) and Rawlsian Egalitarian Welfare Evaluation.
4. **Job 4 (15 min):** MobileNetV3 Edge Footprint Profiling and Multi-Agent Continuous Sensor Regression (^2 = 99.95\%$).

Outputs automatically persist and are downloadable as publication-ready LaTeX tables and figures.

## 0. Mount Google Drive for Persistence across Colab Restarts

In [ ]:
import os, sys
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_KAGGLE:
    print('🚀 Platform: Kaggle (30h Free GPU). All artifacts automatically save to /kaggle/working/outputs.')
elif IS_COLAB:
    print('🚀 Platform: Google Colab. Checking optional Drive mount...')
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        print('✅ Google Drive mounted successfully!')
    except Exception as e:
        print(f'ℹ️ Google Drive mount skipped ({e}). Using local high-speed storage.')
else:
    print('🚀 Platform: Standard Local/Server.')


## 1. Clone Codebase (or Pull Latest Commits)

In [ ]:
import os, pathlib
REPO = 'https://github.com/nam200718/Topology-aware-FDL.git'
ROOT = '/kaggle/working/Topology-aware-FDL' if os.path.exists('/kaggle/working') else '/content/Topology-aware-FDL'
if not pathlib.Path(ROOT).exists():
    !git clone https://github.com/nam200718/Topology-aware-FDL.git {ROOT}
%cd {ROOT}
!git pull origin main
!git log --oneline -3


## 2. Verify GPU Acceleration & Install Requirements

In [ ]:
!pip -q install -r requirements.txt pytest pytest-xdist
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Symlink Google Drive to `outputs/` (Zero-Loss Checkpointing)

In [ ]:
import os, pathlib
if os.path.exists('/kaggle/working'):
    KAGGLE_OUT = '/kaggle/working/outputs'
    os.makedirs(KAGGLE_OUT, exist_ok=True)
    if not os.path.islink('outputs'):
        !rm -rf outputs 2>/dev/null; ln -s /kaggle/working/outputs outputs
        print('✅ Kaggle Output link created: outputs -> /kaggle/working/outputs')
elif pathlib.Path('/content/drive/MyDrive').exists():
    DRIVE_OUT = '/content/drive/MyDrive/aamas_outputs'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    if os.path.exists('outputs') and not os.path.islink('outputs'):
        !mv outputs outputs_local 2>/dev/null
    if not os.path.islink('outputs'):
        !ln -s /content/drive/MyDrive/aamas_outputs outputs
        print('✅ Colab Drive link created: outputs -> /content/drive/MyDrive/aamas_outputs')
else:
    os.makedirs('outputs', exist_ok=True)
    print('ℹ️ Using local outputs directory.')


## 4. Run Unit Verification Suite (15 seconds)

In [ ]:
!rm -rf data/MNIST 2>/dev/null || true
!pytest tests/ -n 4 --ignore=tests/test_cifar.py -v
print('All core and defense components verified successfully!')


## 5. Execute Job 1: CIFAR-100 5 Regimes across Partition Values (~35 mins)
Evaluates FedAvg, FedRep, Ditto, and Defended H-ResFL across IID, Mild ($\alpha=1.0$), Moderate ($\alpha=0.5$), Severe ($\alpha=0.1$), and Extreme ($\alpha=0.05$) Non-IID on CIFAR-100 ($C=100$).

In [ ]:
import os
ROOT = "/kaggle/working/Topology-aware-FDL" if os.path.exists("/kaggle/working/Topology-aware-FDL") else "/content/Topology-aware-FDL"
if os.path.exists(ROOT): %cd {ROOT}
!python scripts/run_aamas_suite.py --job 1
print("Job 1 completed!")

## 6. Execute Job 2: CIFAR-100 Byzantine Multi-Attack Robustness Suite (~25 mins)
Evaluates label-flipping ($y \to 99 - y$) and gradient sign-flipping at attacker fractions $q \in \{0.0, 0.1, 0.2, 0.3\}$, validating Skew-Calibrated Subspace Cosine Defense on 100 classes.

In [ ]:
import os
ROOT = "/kaggle/working/Topology-aware-FDL" if os.path.exists("/kaggle/working/Topology-aware-FDL") else "/content/Topology-aware-FDL"
if os.path.exists(ROOT): %cd {ROOT}
!python scripts/run_aamas_suite.py --job 2
print("Job 2 completed!")

## 7. Execute Job 3: 50-Client Scalability & Rawlsian Egalitarian Welfare (~20 mins)
Validates scalability under 50-client populations with partial participation ($C_p=0.20$), computing Rawlsian min-agent welfare and tail fairness.

In [ ]:
import os
ROOT = "/kaggle/working/Topology-aware-FDL" if os.path.exists("/kaggle/working/Topology-aware-FDL") else "/content/Topology-aware-FDL"
if os.path.exists(ROOT): %cd {ROOT}
!python scripts/run_aamas_suite.py --job 3
print("Job 3 completed!")

## 8. Execute Job 4: MobileNetV3 Edge Footprint & Continuous Sensor Regression (~15 mins)
Profiles edge latency, VRAM, and energy on MobileNetV3-Small, followed by zero-shot task generalization to continuous multi-agent UAV torque regression ($R^2 = 99.95\%$).

In [ ]:
import os
ROOT = "/kaggle/working/Topology-aware-FDL" if os.path.exists("/kaggle/working/Topology-aware-FDL") else "/content/Topology-aware-FDL"
if os.path.exists(ROOT): %cd {ROOT}
!python scripts/run_aamas_suite.py --job 4
print("Job 4 completed!")

## 9. Assemble All Publication-Grade LaTeX Tables & Figures
Compiles the CIFAR-100 5-regime benchmark table, Byzantine robustness matrix, hardware profiling comparisons, and high-DPI figures.

In [ ]:
import os
ROOT = "/kaggle/working/Topology-aware-FDL" if os.path.exists("/kaggle/working/Topology-aware-FDL") else "/content/Topology-aware-FDL"
if os.path.exists(ROOT): %cd {ROOT}
!python scripts/run_aamas_suite.py --job finalize
!ls -lh outputs/*.json
!ls -lh report/figures/*.png 2>/dev/null || true

## 10. Download Complete Results Zip (Fallback if Drive is not mounted)

In [ ]:
!zip -r /tmp/aamas_paper_artifacts.zip outputs/ report/ 2>/dev/null | tail -1
if os.path.exists('/kaggle/working'):
    !cp /tmp/aamas_paper_artifacts.zip /kaggle/working/aamas_paper_artifacts.zip
    print('✅ Saved to Kaggle Output: /kaggle/working/aamas_paper_artifacts.zip is ready to download!')
else:
    try:
        from google.colab import files
        files.download('/tmp/aamas_paper_artifacts.zip')
        print('✅ Download initiated in browser!')
    except Exception as e:
        print(f'Artifact zip ready at: /tmp/aamas_paper_artifacts.zip ({e})')
